# Supervisor Multi-Agent Framework for Scientific Paper Review

This notebook demonstrates a **3-agent supervisor framework** implemented in three different SDKs.  
The same workflow is used in each:

```
              PMID (input)
                  │
                  ▼
    ┌──────────────────────────────┐
    │   Literature Summary Agent   │  (smaller/faster model)
    │   ┌──────────────────────┐  │
    │   │ get_full_text(pmid)  │  │  ← PubMed tool call
    │   └──────────────────────┘  │
    │   Significance · Novelty     │
    │   Technical Soundness        │
    └──────────────┬───────────────┘
                   │ summary
                   ▼
    ┌──────────────────────────────┐
    │       Critic Agent           │  (larger/stronger model)
    │  Reviews the summary as an   │
    │  independent peer reviewer   │
    └──────────────┬───────────────┘
                   │ critique of summary
                   ▼
    ┌──────────────────────────────┐
    │       Supervisor Agent       │
    │  Synthesizes both into the   │
    │  final structured report     │
    └──────────────────────────────┘
```

The literature summary agent **fetches the paper itself** via a PubMed tool call — the input
to the system is a PMID, not pre-loaded text. The critic agent then reviews the summary as an
independent peer reviewer using a stronger model.

| SDK | Package | Supervisor pattern |
|---|---|---|
| **OpenAI Agents SDK** | `agents` (openai-agents) | `Agent.as_tool()` — summary agent has `get_full_text` tool |
| **Anthropic SDK** | `anthropic` | Tool-use loop inside `run_summary_agent` |
| **LangGraph** | `langgraph` + `langchain-anthropic` | `create_react_agent` for the summary node |

## Load Paper from PubMed

`fetch_paper(pmid)` attempts to retrieve the **full text** from PubMed Central (PMC) for
open-access papers, and falls back to the **abstract** for everything else.
Set `PMID` below to any paper you want to review. All three SDK sections use `PAPER_TEXT`
defined here, and the LangGraph section also accepts `PMID` directly as graph input.

In [1]:
import os
import xml.etree.ElementTree as ET
from Bio import Entrez

Entrez.email = os.getenv("NCBI_EMAIL", "your-email@example.com")


def fetch_paper(pmid: str) -> str:
    """Return full body text (PMC open-access) or abstract for a given PMID."""
    links = Entrez.read(Entrez.elink(dbfrom="pubmed", db="pmc", id=pmid))
    pmc_ids = (
        [link["Id"] for link in links[0]["LinkSetDb"][0]["Link"]]
        if links and links[0].get("LinkSetDb")
        else []
    )

    if pmc_ids:
        with Entrez.efetch(db="pmc", id=pmc_ids[0], rettype="full", retmode="xml") as h:
            root = ET.fromstring(h.read())
        sections = []
        for title_el in root.iter("article-title"):
            if title_el.text:
                sections.append("Title: " + title_el.text.strip())
                break
        for abstract_el in root.iter("abstract"):
            text = " ".join(t.strip() for t in abstract_el.itertext() if t.strip())
            if text:
                sections.append("Abstract:\n" + text)
                break
        for sec in root.iter("sec"):
            heading_el = sec.find("title")
            heading = heading_el.text.strip() if heading_el is not None and heading_el.text else ""
            paras = [
                " ".join(t.strip() for t in p.itertext() if t.strip())
                for p in sec.findall("p")
            ]
            paras = [p for p in paras if p]
            if paras:
                sections.append((f"\n{heading}:\n" if heading else "\n") + "\n".join(paras))
        if sections:
            return "\n\n".join(sections)

    # Fall back to abstract (works for all PubMed papers)
    with Entrez.efetch(db="pubmed", id=pmid, rettype="abstract", retmode="text") as h:
        return "[Abstract only — full text not in PMC]\n\n" + h.read()


# ── Set the paper to review ──────────────────────────────────────────────────
PMID = "34265844"   # AlphaFold2: Jumper et al. 2021, Nature
# PMID = "41735549" # Agentic AI in biomedical research: Li et al. 2026, Nature Biotechnology

---
## Section 1 — OpenAI Agents SDK

The summary agent is given `get_full_text` as a callable tool. When the supervisor calls it with
a PMID, the agent decides to invoke the tool, fetches the paper, then produces the summary — all
within a single `Runner.run()` call.

**Pattern**: Supervisor → calls `literature_summary_agent(PMID)` → agent calls `get_full_text(PMID)` → summarizes → calls `critic_agent(summary)` → synthesizes

| Agent | Model | Tools |
|---|---|---|
| Literature Summary | `gpt-4o-mini` | `get_full_text` |
| Critic | `gpt-4o` | — |
| Supervisor | `gpt-4o-mini` | `literature_summary_agent`, `critic_agent` |

In [11]:
import os
from agents import Agent, Runner, function_tool
from dotenv import load_dotenv

load_dotenv()

True

In [12]:
# ── PubMed tool for the summary agent ───────────────────────────────────────
_MAX_PAPER_CHARS = 24_000   # ~6k tokens; enough for a full methods + results section

@function_tool
def get_full_text(pmid: str) -> str:
    """Fetch full article text for a PubMed paper by PMID.
    Returns full body text for open-access PMC papers, or the abstract for all others.
    Text is truncated to keep the response within a manageable size."""
    try:
        text = fetch_paper(pmid)
        if len(text) > _MAX_PAPER_CHARS:
            text = text[:_MAX_PAPER_CHARS] + "\n\n[Truncated]"
        return text
    except Exception as e:
        return f"Could not fetch PMID {pmid}: {type(e).__name__}: {e}"


# ── Literature Summary Agent ─────────────────────────────────────────────────
summary_agent = Agent(
    name="literature-summary-agent",
    handoff_description="Fetches a paper from PubMed and summarizes its significance, novelty, and technical soundness",
    instructions="""You are an expert scientific reviewer specializing in computational biology.
When given a PMID, use the get_full_text tool to fetch the paper, then provide a structured analysis:

**Significance**: Why does this work matter to the field? What problem does it solve?
**Novelty**: What is genuinely new compared to prior work? Is the contribution incremental or transformative?
**Technical Soundness**: Are the methods rigorous? Are the results well-validated and reproducible?

Be specific and cite details from the paper. Keep each section to 3-5 sentences.""",
    tools=[get_full_text],
    model="gpt-4o-mini",
)

# ── Critic Agent ──────────────────────────────────────────────────────────────
# Uses a stronger model to independently challenge the summary agent's assessment.
critic_agent = Agent(
    name="critic-agent",
    handoff_description="Critically reviews a literature summary as an independent peer reviewer",
    instructions="""You are a senior independent peer reviewer. You will receive a structured summary
written by another reviewer about a scientific paper. Your job is to critically assess that summary:

- Challenge claims about significance — is the impact overstated?
- Question novelty assertions — has similar work been done before?
- Probe technical soundness — are there methodological gaps or missing validation?
- Flag limitations the summary glosses over
- Note any reproducibility or accessibility concerns

Return your critique as a bulleted list of 4-6 specific, constructive points directed at the summary.""",
    model="gpt-4o",
)

# ── Supervisor Agent ──────────────────────────────────────────────────────────
supervisor_openai = Agent(
    name="supervisor",
    instructions="""You are a senior scientific editor coordinating a structured paper review.

Given a PMID, follow these steps IN ORDER:
1. Call `literature_summary_agent` with the PMID — it will fetch the paper and return a structured summary.
2. Call `critic_agent` with the summary text returned in step 1.
3. Synthesize both responses into a final structured review report with four clearly labeled sections:
   ## Significance
   ## Novelty
   ## Technical Soundness
   ## Major Critiques

End with a one-paragraph **Overall Assessment**.""",
    tools=[
        summary_agent.as_tool(
            tool_name="literature_summary_agent",
            tool_description="Fetches the paper for a given PMID and summarizes its significance, novelty, and technical soundness.",
        ),
        critic_agent.as_tool(
            tool_name="critic_agent",
            tool_description="Critically reviews a literature summary as an independent peer reviewer. Pass the summary text from literature_summary_agent.",
        ),
    ],
    model="gpt-4o-mini",
)

In [13]:
# ── Sanity check: test the underlying fetch logic directly ─────────────────
# get_full_text is now a FunctionTool (not callable); test fetch_paper instead.
test_result = fetch_paper(PMID)
if len(test_result) > _MAX_PAPER_CHARS:
    test_result = test_result[:_MAX_PAPER_CHARS] + '\n\n[Truncated]'
print(f'fetch_paper returned {len(test_result):,} chars')
print(test_result[:300], '...')

fetch_paper returned 24,013 chars
Title: Highly accurate protein structure prediction with AlphaFold

Abstract:
Proteins are essential to life, and understanding their structure can facilitate a mechanistic understanding of their function. Through an enormous experimental effort 1 – 4 , the structures of around 100,000 unique protei ...


In [14]:
from agents import RunHooks

class DebugHooks(RunHooks):
    async def on_agent_start(self, context, agent):
        print(f'[→ AGENT] {agent.name}')

    async def on_agent_end(self, context, agent, output):
        out_str = str(output)[:120].replace('\n', ' ')
        print(f'[← AGENT] {agent.name}: {out_str!r}')

    async def on_tool_start(self, context, agent, tool):
        print(f'  [→ TOOL] {agent.name} calls {tool.name}')

    async def on_tool_end(self, context, agent, tool, result):
        result_str = str(result)[:120].replace('\n', ' ')
        print(f'  [← TOOL] {tool.name}: {result_str!r}')

result_openai = await Runner.run(
    supervisor_openai,
    f'Review the paper with PMID {PMID}',
    max_turns=25,
    hooks=DebugHooks(),
)

print('=' * 70)
print('OPENAI AGENTS SDK — PAPER REVIEW')
print('=' * 70)
print(result_openai.final_output)

[→ AGENT] supervisor
  [→ TOOL] supervisor calls literature_summary_agent
  [← TOOL] literature_summary_agent: '### Significance The work presented in this paper is significant as it addresses a longstanding challenge in computation'
  [→ TOOL] supervisor calls critic_agent
  [← TOOL] critic_agent: "- **Significance Overstated**: While AlphaFold's achievements in structure prediction are indeed significant, the summar"
[← AGENT] supervisor: '## Significance The work presented in this paper is significant as it addresses a longstanding challenge in computationa'
OPENAI AGENTS SDK — PAPER REVIEW
## Significance
The work presented in this paper is significant as it addresses a longstanding challenge in computational biology: the accurate prediction of protein structures from amino acid sequences, an endeavor critical for understanding protein function and interactions. While AlphaFold enables large-scale structural bioinformatics and has the potential to transform how researchers approach

---
## Section 2 — Anthropic SDK (Direct Python Orchestration)

`run_summary_agent` runs an Anthropic **tool-use loop**: it gives Claude the `get_full_text`
tool definition, then executes tool calls by routing them to `fetch_paper()` until the model
produces a final text response. This is the same pattern as the OpenAI Agents SDK, but written
explicitly in Python rather than managed by an SDK runner.

**Pattern**: `run_supervisor(pmid)` → `run_summary_agent(pmid)` [tool loop] → `run_critic_agent(summary)` → synthesis

| Agent | Model | Tools |
|---|---|---|
| Literature Summary | `claude-sonnet-4-6` | `get_full_text` (tool-use loop) |
| Critic | `claude-opus-4-7` | — |
| Supervisor | `claude-sonnet-4-6` | synthesizes both outputs |

In [15]:
import anthropic
from dotenv import load_dotenv

load_dotenv()
# Requires ANTHROPIC_API_KEY in environment

client = anthropic.Anthropic()

In [16]:
SUMMARY_MODEL = "claude-sonnet-4-6"
CRITIC_MODEL = "claude-opus-4-7"

_PUBMED_TOOLS = [
    {
        "name": "get_full_text",
        "description": (
            "Fetch full article text for a PubMed paper by PMID. "
            "Returns full body text for open-access PMC papers, or the abstract for all others."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"pmid": {"type": "string", "description": "PubMed ID"}},
            "required": ["pmid"],
        },
    }
]


def run_summary_agent(pmid: str) -> str:
    """Literature Summary Agent — fetches paper via tool call, then summarizes."""
    messages = [{"role": "user", "content": f"Fetch and analyze the paper with PMID {pmid}."}]

    while True:
        response = client.messages.create(
            model=SUMMARY_MODEL,
            max_tokens=2048,
            system="""You are an expert scientific reviewer specializing in computational biology.
Use get_full_text to fetch the paper, then provide a structured analysis with three sections:

**Significance**: Why does this work matter to the field? What problem does it solve?
**Novelty**: What is genuinely new compared to prior work? Is the contribution incremental or transformative?
**Technical Soundness**: Are the methods rigorous? Are the results well-validated and reproducible?

Be specific and cite details from the paper. Keep each section to 3-5 sentences.""",
            tools=_PUBMED_TOOLS,
            messages=messages,
        )

        if response.stop_reason == "end_turn":
            return next((b.text for b in response.content if hasattr(b, "text")), "")

        # Execute tool calls and feed results back
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use" and block.name == "get_full_text":
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": fetch_paper(block.input["pmid"]),
                })
        messages.append({"role": "user", "content": tool_results})


def run_critic_agent(summary: str) -> str:
    """Critic Agent — independently challenges the summary agent's assessment."""
    response = client.messages.create(
        model=CRITIC_MODEL,
        max_tokens=1024,
        system="""You are a senior independent peer reviewer. You will receive a structured summary
written by another reviewer about a scientific paper. Your job is to critically assess that summary:

- Challenge claims about significance — is the impact overstated?
- Question novelty assertions — has similar work been done before?
- Probe technical soundness — are there methodological gaps or missing validation?
- Flag limitations the summary glosses over
- Note any reproducibility or accessibility concerns

Return your critique as a bulleted list of 4-6 specific, constructive points directed at the summary.""",
        messages=[{"role": "user", "content": summary}],
    )
    return response.content[0].text


def run_supervisor(pmid: str) -> str:
    """Supervisor — orchestrates the two agents and synthesizes a final review report."""
    summary = run_summary_agent(pmid)
    critique = run_critic_agent(summary)

    synthesis_prompt = f"""You received the following two expert reviews.

--- LITERATURE SUMMARY (by summary agent) ---
{summary}

--- INDEPENDENT CRITIQUE (by critic agent, reviewing the summary) ---
{critique}

Synthesize these into a final structured review report with four labeled sections:
## Significance
## Novelty
## Technical Soundness
## Major Critiques

End with a one-paragraph **Overall Assessment**."""

    response = client.messages.create(
        model=SUMMARY_MODEL,
        max_tokens=2048,
        system="You are a senior scientific editor synthesizing peer review feedback into a coherent report.",
        messages=[{"role": "user", "content": synthesis_prompt}],
    )
    return response.content[0].text

In [ ]:
result_anthropic = run_supervisor(PMID)

print("=" * 70)
print("ANTHROPIC SDK — PAPER REVIEW")
print("=" * 70)
print(result_anthropic)

---
## Section 3 — LangGraph (LangChain Ecosystem)

The summary node uses `create_react_agent` — a LangGraph prebuilt that runs a tool-use loop
internally. The `pubmed_get_full_text` LangChain tool is registered on it; the agent calls it
when it receives a PMID. The critic and supervisor nodes remain plain LLM calls.

**Pattern**: `START` → `summary_agent` (ReAct, calls `pubmed_get_full_text`) → `critic_agent` (reads `state["summary"]`) → `supervisor` → `END`

| Node | Model | Implementation |
|---|---|---|
| `summary_agent` | `claude-sonnet-4-6` | `create_react_agent` with `pubmed_get_full_text` tool |
| `critic_agent` | `claude-opus-4-7` | plain LLM call on `state["summary"]` |
| `supervisor` | `claude-sonnet-4-6` | plain LLM call synthesizing both |

> **Note**: The DeepAgents SDK (`deepagents`) is a thin wrapper around LangGraph that adds built-in
> subagent spawning and task planning. The LangGraph implementation below is equivalent to what
> DeepAgents produces under the hood, with explicit control over graph structure.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool as lc_tool
from dotenv import load_dotenv

load_dotenv()
# Requires ANTHROPIC_API_KEY in environment

llm_summary = ChatAnthropic(model="claude-haiku-4-5", max_tokens=2048)
llm_critic = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024)

In [ ]:
class ReviewState(TypedDict):
    pmid: str        # input: PubMed ID
    summary: str
    critique: str
    final_review: str


# ── PubMed tool for the summary agent ──────────────────────────────────────
@lc_tool
def pubmed_get_full_text(pmid: str) -> str:
    """Fetch full article text for a PubMed paper by PMID.
    Returns full body text for open-access PMC papers, or the abstract for all others."""
    return fetch_paper(pmid)   # reuses fetch_paper() from the setup cell


# ── Node: Literature Summary Agent (ReAct) ─────────────────────────────────
_SUMMARY_PROMPT = SystemMessage(content="""You are an expert scientific reviewer specializing in computational biology.
When given a PMID, use pubmed_get_full_text to fetch the paper, then provide a structured analysis:

**Significance**: Why does this work matter to the field? What problem does it solve?
**Novelty**: What is genuinely new compared to prior work? Is the contribution incremental or transformative?
**Technical Soundness**: Are the methods rigorous? Are the results well-validated and reproducible?

Be specific and cite details from the paper. Keep each section to 3-5 sentences.""")

_summary_react = create_react_agent(llm_summary, [pubmed_get_full_text], prompt=_SUMMARY_PROMPT)

def summary_node(state: ReviewState) -> dict:
    result = _summary_react.invoke({
        "messages": [HumanMessage(content=f"Fetch and summarize the paper with PMID {state['pmid']}.")]
    })
    return {"summary": result["messages"][-1].content}


# ── Node: Critic Agent ─────────────────────────────────────────────────────
# Receives the summary (not the raw paper) and uses a stronger model to challenge it.
def critic_node(state: ReviewState) -> dict:
    response = llm_critic.invoke([
        SystemMessage(content="""You are a senior independent peer reviewer. You will receive a structured summary
written by another reviewer about a scientific paper. Your job is to critically assess that summary:

- Challenge claims about significance — is the impact overstated?
- Question novelty assertions — has similar work been done before?
- Probe technical soundness — are there methodological gaps or missing validation?
- Flag limitations the summary glosses over
- Note any reproducibility or accessibility concerns

Return your critique as a bulleted list of 4-6 specific, constructive points directed at the summary."""),
        HumanMessage(content=state["summary"]),
    ])
    return {"critique": response.content}


# ── Node: Supervisor (synthesis) ───────────────────────────────────────────
def supervisor_node(state: ReviewState) -> dict:
    synthesis_prompt = f"""You received the following two expert reviews.

--- LITERATURE SUMMARY (by summary agent) ---
{state['summary']}

--- INDEPENDENT CRITIQUE (by critic agent, reviewing the summary) ---
{state['critique']}

Synthesize these into a final structured review report with four labeled sections:
## Significance
## Novelty
## Technical Soundness
## Major Critiques

End with a one-paragraph **Overall Assessment**."""

    response = llm_summary.invoke([
        SystemMessage(content="You are a senior scientific editor synthesizing peer review feedback into a coherent report."),
        HumanMessage(content=synthesis_prompt),
    ])
    return {"final_review": response.content}


# ── Build Graph ────────────────────────────────────────────────────────────
builder = StateGraph(ReviewState)
builder.add_node("summary_agent", summary_node)
builder.add_node("critic_agent", critic_node)
builder.add_node("supervisor", supervisor_node)

builder.set_entry_point("summary_agent")
builder.add_edge("summary_agent", "critic_agent")
builder.add_edge("critic_agent", "supervisor")
builder.add_edge("supervisor", END)

workflow = builder.compile()

In [ ]:
result_langgraph = workflow.invoke({"pmid": PMID})

print("=" * 70)
print("LANGGRAPH — PAPER REVIEW")
print("=" * 70)
print(result_langgraph["final_review"])

In [ ]:
# Optional: view intermediate agent outputs
print("── Summary Agent Output ──")
print(result_langgraph["summary"])
print()
print("── Critic Agent Output ──")
print(result_langgraph["critique"])

---
## Comparison

| Dimension | OpenAI Agents SDK | Anthropic SDK | LangGraph |
|---|---|---|---|
| **Workflow control** | LLM decides tool order | Python controls order | Graph edges enforce order |
| **Intermediate results** | Hidden inside Runner | Explicit Python variables | Stored in shared State dict |
| **Flexibility** | LLM can skip/reorder tools | Fixed sequence | Fixed graph topology |
| **Debugging** | Via tracing (OpenAI SDK) | Print/log directly | Log per-node state |
| **Model** | OpenAI models (or litellm) | Any Anthropic model | Any LangChain-supported model |
| **Best for** | Autonomous tool use | Simple, predictable pipelines | Complex branching workflows |